# AutoMark – train the handwriting model on SSL-HWD-Words (Kaggle)

Model: our own reader, **CNN (5 layers) + BiLSTM (2 layers) + CTC**, the same network as `grader/htr_model.py`.
It starts from the current app weights (`model/htr_weights.npz`, trained on IAM) and is fine-tuned on
a sample of the **labelled** part of SSL-HWD-Words (Hugging Face `Shree10/SSL-HWD-Words`, Apache-2.0).

Run the cells from top to bottom. Everything this notebook produces is saved in `/kaggle/working/results/`:

| File | What it holds |
|---|---|
| `train_loss.txt` | training CTC loss, one line per epoch |
| `val_loss.txt` | validation CTC loss, one line per epoch |
| `val_cer.txt` | validation character error rate (%) per epoch |
| `val_word_accuracy.txt` | validation words fully correct (%) per epoch |
| `grad_norm.txt` | average / max gradient norm per epoch |
| `grad_norm_per_layer.txt` | average gradient norm of every layer per epoch |
| `learning_rate.txt` | learning rate per epoch |
| `epoch_time.txt` | seconds per epoch |
| `history.csv` | all of the above in one table |
| `*.png` | graphs: loss, CER, word accuracy, gradient norms, learning rate |
| `test_results.txt` | final test: before vs after training (+ your classmates, if added) |
| `sample_predictions.txt` | examples: correct text vs prediction before and after |
| `dataset_info.txt`, `config.txt` | what data and settings were used |
| `htr_weights_best.npz` | **best model in the app's format** → copy to `automark/model/htr_weights.npz` |
| `best_model.pt`, `last_checkpoint.pt` | PyTorch checkpoints |
| `results.zip` | everything above in one file to download |


## 1. Settings
Change numbers here only. The defaults fit one Kaggle GPU session (12 h).

In [ ]:
# ---------------- data
N_TRAIN_WORDS   = 100_000   # words used for training (more = slower, usually a bit better)
N_VAL_WORDS     = 10_000    # words used to pick the best epoch
N_TEST_WORDS    = 10_000    # words only used once, at the end
MAX_PER_WRITER  = 300       # at most this many words from one writer (keeps many different writers)
MAX_LABEL_LEN   = 24        # longer words are skipped (the model has 32 time steps)

# ---------------- training
EPOCHS          = 30
BATCH_SIZE      = 64
LEARNING_RATE   = 1e-4      # small: we fine-tune an already trained model
PATIENCE        = 6         # stop when validation CER has not improved for this many epochs
START_FROM_APP_WEIGHTS = True   # False = train from random weights (needs far more data and time)
SEED            = 42

# ---------------- paths
HF_REPO   = "Shree10/SSL-HWD-Words"
HF_FILE   = "SSL-HWD-Words(Labeled).zip"            # 14.5 GB, labelled words only
DATA_DIR  = "/tmp/sslhwd"                           # download place (not saved with the notebook)
GITHUB    = "https://github.com/demonayush11/Automark.git"
OUT       = "/kaggle/working/results"

import os
os.makedirs(OUT, exist_ok=True)
with open(f"{OUT}/config.txt", "w") as f:
    for k, v in dict(globals()).items():
        if k.isupper():
            f.write(f"{k} = {v}\n")
print("Settings saved to", f"{OUT}/config.txt")


## 2. Check the GPU and install the downloader
Should print `cuda` and a GPU name. If it prints `cpu`: right panel → **Session options → Accelerator → GPU T4 x2**.

In [ ]:
!pip -q install -U huggingface_hub hf_transfer
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "NO GPU - training will be very slow")


## 3. Get the current model weights
Downloads the AutoMark code from GitHub. If that fails (private repo), upload `htr_weights.npz` and `charList.txt` as a Kaggle dataset (**+ Add Input → Upload**); this cell then finds them.

In [ ]:
import glob, subprocess
if not os.path.exists("/kaggle/working/automark"):
    subprocess.run(["git", "clone", "-q", GITHUB, "/kaggle/working/automark"])

def find(name):
    hits = sorted(glob.glob(f"/kaggle/working/automark/**/{name}", recursive=True)) + \
           sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"{name} not found: clone failed and no Kaggle dataset has it"
    return hits[0]

WEIGHTS = find("htr_weights.npz")
CHARS = open(find("charList.txt"), encoding="utf-8").read()
print("Weights:", WEIGHTS)
print("Characters the model knows:", len(CHARS))


## 4. Download the labelled dataset (14.5 GB)
Takes about 5–20 minutes on Kaggle. If the dataset asks for a login, add your Hugging Face token: **Add-ons → Secrets → `HF_TOKEN`**.

In [ ]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
ZIP_PATH = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE, repo_type="dataset",
                           local_dir=DATA_DIR, token=token)
print("Downloaded:", ZIP_PATH, round(os.path.getsize(ZIP_PATH) / 1e9, 2), "GB")
!df -h /tmp | tail -1


## 5. Look inside the zip
Prints the folder layout and one label file, so you can see the data is right.

In [ ]:
import zipfile, posixpath, io, collections
import pandas as pd

zf = zipfile.ZipFile(ZIP_PATH)
NAMES = [n for n in zf.namelist() if not n.startswith("__MACOSX") and not n.endswith("/")]
CSVS = sorted(n for n in NAMES if n.lower().endswith(".csv"))
IMAGES = [n for n in NAMES if n.lower().endswith((".png", ".jpg", ".jpeg"))]
print(f"{len(NAMES):,} files: {len(IMAGES):,} images, {len(CSVS):,} label files")
top = collections.Counter("/".join(n.split("/")[:2]) for n in NAMES)
print("Top folders:", list(top.most_common(8)))
print("Example label files:", CSVS[:3])
print(pd.read_csv(io.BytesIO(zf.read(CSVS[0]))).head())


## 6. Pick the words and load them
Writers are split into **train / validation / test**, so the model is always tested on writers it never saw. Takes about 5–15 minutes.

In [ ]:
import random, cv2, numpy as np
from tqdm.auto import tqdm

random.seed(SEED); np.random.seed(SEED)
CHARSET = set(CHARS)
NAME_SET = set(NAMES)
_by_base = None
def resolve(folder, fname):
    """Path of an image inside the zip, from the label file's folder and the file name."""
    global _by_base
    p = posixpath.normpath(posixpath.join(folder, fname))
    if p in NAME_SET:
        return p
    if _by_base is None:                               # build a name -> path index once, only if needed
        _by_base = {}
        for n in IMAGES:
            _by_base.setdefault(posixpath.basename(n), n)
    return _by_base.get(posixpath.basename(fname))

def read_labels(csv_path):
    df = pd.read_csv(io.BytesIO(zf.read(csv_path)), dtype=str, keep_default_na=False)
    cols = {c.lower().strip(): c for c in df.columns}
    fcol = cols.get("image_filename", df.columns[0])
    tcol = cols.get("transcription", df.columns[1])
    rows = []
    for f, t in zip(df[fcol], df[tcol]):
        t = str(t).strip()
        if 1 <= len(t) <= MAX_LABEL_LEN and all(c in CHARSET for c in t):
            rows.append((f, t))
    return rows

def load_gray(path):
    img = cv2.imdecode(np.frombuffer(zf.read(path), np.uint8), cv2.IMREAD_GRAYSCALE)
    if img is None or img.size == 0:
        return None
    if img.mean() < 110:                              # light text on dark background -> invert
        img = 255 - img
    h, w = img.shape
    s = min(1.0, 64 / h, 512 / w)                     # store small to save memory
    if s < 1:
        img = cv2.resize(img, (max(1, int(w * s)), max(1, int(h * s))), interpolation=cv2.INTER_AREA)
    return img

csv_order = CSVS[:]
random.shuffle(csv_order)
splits = {"test": [], "val": [], "train": []}
goal = {"test": N_TEST_WORDS, "val": N_VAL_WORDS, "train": N_TRAIN_WORDS}
writers = {"test": [], "val": [], "train": []}
skipped_labels = 0
bar = tqdm(total=sum(goal.values()), desc="words loaded")
for csv_path in csv_order:
    part = next((k for k in ("test", "val", "train") if len(splits[k]) < goal[k]), None)
    if part is None:
        break
    folder = posixpath.dirname(csv_path)
    rows = read_labels(csv_path)
    random.shuffle(rows)
    took = 0
    for fname, text in rows:
        if took >= MAX_PER_WRITER or len(splits[part]) >= goal[part]:
            break
        p = resolve(folder, fname)
        img = load_gray(p) if p else None
        if img is None:
            skipped_labels += 1
            continue
        splits[part].append((img, text))
        took += 1
        bar.update(1)
    if took:
        writers[part].append(folder)
bar.close()

with open(f"{OUT}/dataset_info.txt", "w") as f:
    f.write(f"Dataset: {HF_REPO} / {HF_FILE}\n")
    f.write(f"Label files in zip: {len(CSVS)}; images in zip: {len(IMAGES)}\n")
    for k in ("train", "val", "test"):
        line = f"{k:5}: {len(splits[k]):7,} words from {len(writers[k]):4} writers"
        print(line); f.write(line + "\n")
    f.write(f"Rows skipped (image missing/unreadable): {skipped_labels}\n")
    f.write(f"Labels kept only if 1-{MAX_LABEL_LEN} characters, all among the model's {len(CHARS)} characters\n")
    f.write("Writers never overlap between train, val and test.\n")
assert splits["train"] and splits["val"] and splits["test"], "Not enough data found - check cell 5 output"

# a picture of 24 random training words with their labels (for the report)
import matplotlib.pyplot as plt
fig, axes = plt.subplots(6, 4, figsize=(12, 9))
for ax, (img, text) in zip(axes.flat, random.sample(splits["train"], 24)):
    ax.imshow(img, cmap="gray"); ax.set_title(text, fontsize=10); ax.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/sample_training_words.png", dpi=110); plt.show()


## 7. The model (same network as the app) and weight conversion

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
IMG_W, IMG_H, T_STEPS, HIDDEN = 128, 32, 32, 256

class HTR(nn.Module):
    """5 x (conv -> batch-norm -> ReLU -> max-pool) -> LSTM read forwards and backwards with the
    SAME weights -> linear layer to 78 classes (77 characters + CTC blank). Same as grader/htr_model.py."""
    def __init__(self, n_classes):
        super().__init__()
        chans, kernels = [1, 32, 64, 128, 128, 256], [5, 5, 3, 3, 3]
        self.pools = [(2, 2), (2, 2), (1, 2), (1, 2), (1, 2)]
        self.convs = nn.ModuleList(nn.Conv2d(chans[i], chans[i + 1], kernels[i], padding=kernels[i] // 2, bias=False)
                                   for i in range(5))
        self.bns = nn.ModuleList(nn.BatchNorm2d(chans[i + 1], eps=1e-3) for i in range(5))
        self.lstm = nn.LSTM(256, HIDDEN, num_layers=2, batch_first=True)
        self.out = nn.Linear(2 * HIDDEN, n_classes, bias=False)

    def forward(self, x):                              # x: (B, 128, 32)
        x = x.unsqueeze(1)
        for conv, bn, pool in zip(self.convs, self.bns, self.pools):
            x = F.max_pool2d(F.relu(bn(conv(x))), pool)
        seq = x.squeeze(3).permute(0, 2, 1)            # (B, 32, 256)
        fw, _ = self.lstm(seq)
        bw, _ = self.lstm(torch.flip(seq, [1]))
        return self.out(torch.cat([fw, torch.flip(bw, [1])], dim=2))   # (B, 32, 78)

LSTM_KEY = "bidirectional_rnn__fw__multi_rnn_cell__cell_{}__lstm_cell__{}"

def load_npz(model, path):
    """App weights (TensorFlow layout) -> PyTorch."""
    w = np.load(path)
    t = lambda a: torch.tensor(np.ascontiguousarray(a), dtype=torch.float32)
    with torch.no_grad():
        for i in range(5):
            model.convs[i].weight.copy_(t(w["Variable" + ("" if i == 0 else f"_{i}")].transpose(3, 2, 0, 1)))
            bn = "batch_normalization" + ("" if i == 0 else f"_{i}")
            model.bns[i].weight.copy_(t(w[bn + "__gamma"])); model.bns[i].bias.copy_(t(w[bn + "__beta"]))
            model.bns[i].running_mean.copy_(t(w[bn + "__moving_mean"]))
            model.bns[i].running_var.copy_(t(w[bn + "__moving_variance"]))
        for layer in range(2):
            kernel, bias = w[LSTM_KEY.format(layer, "kernel")], w[LSTM_KEY.format(layer, "bias")]
            n_in = kernel.shape[0] - HIDDEN
            gi, gj, gf, go = np.split(kernel, 4, axis=1); bi, bj, bf, bo = np.split(bias, 4)
            k = np.concatenate([gi, gf, gj, go], axis=1)
            getattr(model.lstm, f"weight_ih_l{layer}").copy_(t(k[:n_in].T))
            getattr(model.lstm, f"weight_hh_l{layer}").copy_(t(k[n_in:].T))
            getattr(model.lstm, f"bias_ih_l{layer}").copy_(t(np.concatenate([bi, bf + 1.0, bj, bo])))
            getattr(model.lstm, f"bias_hh_l{layer}").zero_()
        model.out.weight.copy_(t(w["Variable_5"][0, 0].T))

def save_npz(model, path):
    """PyTorch -> app weights file (put it in automark/model/htr_weights.npz)."""
    a = lambda p: p.detach().cpu().numpy().astype(np.float32)
    d = {}
    for i in range(5):
        d["Variable" + ("" if i == 0 else f"_{i}")] = a(model.convs[i].weight).transpose(2, 3, 1, 0)
        bn = "batch_normalization" + ("" if i == 0 else f"_{i}")
        d[bn + "__gamma"], d[bn + "__beta"] = a(model.bns[i].weight), a(model.bns[i].bias)
        d[bn + "__moving_mean"], d[bn + "__moving_variance"] = a(model.bns[i].running_mean), a(model.bns[i].running_var)
    for layer in range(2):
        w_ih, w_hh = a(getattr(model.lstm, f"weight_ih_l{layer}")), a(getattr(model.lstm, f"weight_hh_l{layer}"))
        b = a(getattr(model.lstm, f"bias_ih_l{layer}")) + a(getattr(model.lstm, f"bias_hh_l{layer}"))
        gi, gf, gg, go = np.split(np.concatenate([w_ih.T, w_hh.T], axis=0), 4, axis=1)
        bi, bf, bg, bo = np.split(b, 4)
        d[LSTM_KEY.format(layer, "kernel")] = np.concatenate([gi, gg, gf, go], axis=1)
        d[LSTM_KEY.format(layer, "bias")] = np.concatenate([bi, bg, bf - 1.0, bo])
    d["Variable_5"] = a(model.out.weight).T[None, None]
    np.savez_compressed(path, **d)

model = HTR(len(CHARS) + 1)
if START_FROM_APP_WEIGHTS:
    load_npz(model, WEIGHTS)
    print("Started from the app weights:", WEIGHTS)
model = model.to(DEVICE)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")


## 8. Image preparation, random changes (augmentation) and batches

In [ ]:
from torch.utils.data import Dataset, DataLoader
CHAR_INDEX = {c: i for i, c in enumerate(CHARS)}
BLANK = len(CHARS)

def preprocess(gray):
    """Exactly as the app (grader/htr_model.py): fit into 128x32 on white, transpose, normalise."""
    h, w = gray.shape
    f = max(w / IMG_W, h / IMG_H)
    nw, nh = max(min(IMG_W, int(w / f)), 1), max(min(IMG_H, int(h / f)), 1)
    canvas = np.ones((IMG_H, IMG_W)) * 255
    canvas[:nh, :nw] = cv2.resize(gray, (nw, nh))
    img = cv2.transpose(canvas)
    m, s = img.mean(), img.std()
    return ((img - m) / s if s > 0 else img - m).astype(np.float32)

def augment(gray, rng):
    """Small random changes so the model learns the writing, not the exact pixels."""
    h, w = gray.shape
    gray = cv2.resize(gray, (max(1, int(w * rng.uniform(0.8, 1.2))), h))      # wider / narrower
    h, w = gray.shape
    m = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-3, 3), 1)        # small rotation
    m[0, 1] += rng.uniform(-0.25, 0.25)                                       # slant
    gray = cv2.warpAffine(gray, m, (w, h), borderValue=255)
    r = rng.random()
    if r < 0.25:   gray = cv2.erode(gray, np.ones((2, 2), np.uint8))          # thicker ink
    elif r < 0.5:  gray = cv2.dilate(gray, np.ones((2, 2), np.uint8))         # thinner ink
    if rng.random() < 0.3:
        gray = cv2.GaussianBlur(gray, (3, 3), 0)                              # slight blur
    return gray

class Words(Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        img, text = self.items[i]
        if self.train:
            img = augment(img, np.random.default_rng())
        return torch.from_numpy(preprocess(img)), [CHAR_INDEX[c] for c in text], text

def collate(batch):
    x = torch.stack([b[0] for b in batch])
    targets = torch.tensor([c for b in batch for c in b[1]], dtype=torch.long)
    lengths = torch.tensor([len(b[1]) for b in batch], dtype=torch.long)
    return x, targets, lengths, [b[2] for b in batch]

loader = lambda part, train: DataLoader(Words(splits[part], train), batch_size=BATCH_SIZE, shuffle=train,
                                        num_workers=4, collate_fn=collate, pin_memory=(DEVICE == "cuda"),
                                        persistent_workers=True)
train_loader, val_loader, test_loader = loader("train", True), loader("val", False), loader("test", False)
print("Batches per epoch:", len(train_loader))


## 9. Measuring: loss, character error rate (CER), word accuracy

In [ ]:
ctc_loss = nn.CTCLoss(blank=BLANK, zero_infinity=True)

def decode(logits):
    out = []
    for seq in logits.argmax(2).cpu().numpy():
        txt, prev = [], None
        for k in seq:
            if k != prev and k != BLANK:
                txt.append(CHARS[k])
            prev = k
        out.append("".join(txt))
    return out

def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]

@torch.no_grad()
def evaluate(model, data_loader, keep_examples=0):
    """Returns loss, CER %, word accuracy %, and some (truth, prediction) examples."""
    model.eval()
    loss_sum = n = errs = chars = right = 0
    examples = []
    for x, targets, lengths, texts in data_loader:
        logits = model(x.to(DEVICE))
        logp = F.log_softmax(logits, 2).permute(1, 0, 2)
        loss = ctc_loss(logp, targets, torch.full((len(texts),), T_STEPS, dtype=torch.long), lengths)
        loss_sum += loss.item() * len(texts); n += len(texts)
        for pred, truth in zip(decode(logits), texts):
            errs += edit_distance(pred, truth); chars += len(truth); right += pred == truth
            if len(examples) < keep_examples:
                examples.append((truth, pred))
    return loss_sum / n, 100 * errs / chars, 100 * right / n, examples


## 10. Score BEFORE training (the baseline for your report)

In [ ]:
base_val = evaluate(model, val_loader)
base_test = evaluate(model, test_loader, keep_examples=40)
print(f"Before training  | val: loss {base_val[0]:.3f}  CER {base_val[1]:.2f}%  words correct {base_val[2]:.2f}%")
print(f"Before training  | test: loss {base_test[0]:.3f}  CER {base_test[1]:.2f}%  words correct {base_test[2]:.2f}%")


## 11. Train
After every epoch it prints training loss, validation loss, CER, word accuracy and **gradient norms**, appends
them to the txt files, and saves the model **only when validation CER improved** (`htr_weights_best.npz`).
`last_checkpoint.pt` is saved every epoch. Stops early when validation CER stops improving.

In [ ]:
import time, csv
opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
LAYERS = {f"conv{i+1}": [model.convs[i].weight, model.bns[i].weight, model.bns[i].bias] for i in range(5)}
LAYERS["lstm"] = list(model.lstm.parameters())
LAYERS["output"] = [model.out.weight]

def write(name, text, header=None):
    path = f"{OUT}/{name}"
    new = not os.path.exists(path)
    with open(path, "a") as f:
        if new and header: f.write(header + "\n")
        f.write(text + "\n")

for name in ["train_loss.txt", "val_loss.txt", "val_cer.txt", "val_word_accuracy.txt", "grad_norm.txt",
             "grad_norm_per_layer.txt", "learning_rate.txt", "epoch_time.txt", "history.csv"]:
    if os.path.exists(f"{OUT}/{name}"): os.remove(f"{OUT}/{name}")
# epoch 0 = before training
write("val_loss.txt", f"0\t{base_val[0]:.5f}", "epoch\tval_loss")
write("val_cer.txt", f"0\t{base_val[1]:.3f}", "epoch\tval_cer_percent")
write("val_word_accuracy.txt", f"0\t{base_val[2]:.3f}", "epoch\tval_word_accuracy_percent")

history = []
best_cer, best_epoch = base_val[1], 0
save_npz(model, f"{OUT}/htr_weights_best.npz")          # if nothing improves, "best" = the starting model
torch.save(model.state_dict(), f"{OUT}/best_model.pt")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    loss_sum = n = 0
    grad_norms, layer_sums = [], {k: 0.0 for k in LAYERS}
    for x, targets, lengths, texts in tqdm(train_loader, desc=f"epoch {epoch}", leave=False):
        logits = model(x.to(DEVICE, non_blocking=True))
        logp = F.log_softmax(logits, 2).permute(1, 0, 2)
        loss = ctc_loss(logp, targets, torch.full((len(texts),), T_STEPS, dtype=torch.long), lengths)
        opt.zero_grad()
        loss.backward()
        for k, params in LAYERS.items():                 # gradient size of each layer
            layer_sums[k] += torch.norm(torch.stack([p.grad.norm() for p in params if p.grad is not None])).item()
        grad_norms.append(nn.utils.clip_grad_norm_(model.parameters(), 5.0).item())   # total, before clipping
        opt.step()
        loss_sum += loss.item() * len(texts); n += len(texts)

    train_loss = loss_sum / n
    val_loss, val_cer, val_acc, _ = evaluate(model, val_loader)
    lr = opt.param_groups[0]["lr"]
    sched.step(val_cer)
    secs = time.time() - t0
    g_mean, g_max = float(np.mean(grad_norms)), float(np.max(grad_norms))
    per_layer = {k: v / len(train_loader) for k, v in layer_sums.items()}

    improved = val_cer < best_cer
    if improved:
        best_cer, best_epoch = val_cer, epoch
        save_npz(model, f"{OUT}/htr_weights_best.npz")
        torch.save(model.state_dict(), f"{OUT}/best_model.pt")
    torch.save({"epoch": epoch, "model": model.state_dict(), "optimizer": opt.state_dict(),
                "best_cer": best_cer}, f"{OUT}/last_checkpoint.pt")

    print(f"Epoch {epoch:3d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | val CER {val_cer:.2f}% "
          f"| words correct {val_acc:.2f}% | grad norm mean {g_mean:.3f} max {g_max:.3f} | lr {lr:.1e} "
          f"| {secs/60:.1f} min{'  <- best, saved' if improved else ''}")
    print("          gradient norm per layer: " + "  ".join(f"{k} {v:.4f}" for k, v in per_layer.items()))

    write("train_loss.txt", f"{epoch}\t{train_loss:.5f}", "epoch\ttrain_loss")
    write("val_loss.txt", f"{epoch}\t{val_loss:.5f}")
    write("val_cer.txt", f"{epoch}\t{val_cer:.3f}")
    write("val_word_accuracy.txt", f"{epoch}\t{val_acc:.3f}")
    write("grad_norm.txt", f"{epoch}\t{g_mean:.5f}\t{g_max:.5f}", "epoch\tmean_grad_norm\tmax_grad_norm")
    write("grad_norm_per_layer.txt", f"{epoch}\t" + "\t".join(f"{v:.6f}" for v in per_layer.values()),
          "epoch\t" + "\t".join(per_layer))
    write("learning_rate.txt", f"{epoch}\t{lr:.2e}", "epoch\tlearning_rate")
    write("epoch_time.txt", f"{epoch}\t{secs:.1f}", "epoch\tseconds")
    row = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_cer": val_cer,
           "val_word_accuracy": val_acc, "grad_norm_mean": g_mean, "grad_norm_max": g_max, "lr": lr,
           "seconds": secs, **{f"grad_{k}": v for k, v in per_layer.items()}}
    history.append(row)
    with open(f"{OUT}/history.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row)); w.writeheader(); w.writerows(history)

    if epoch - best_epoch >= PATIENCE:
        print(f"No improvement for {PATIENCE} epochs: stopping.")
        break

print(f"\nBest validation CER {best_cer:.2f}% at epoch {best_epoch} (before training: {base_val[1]:.2f}%)")


## 12. Graphs for the report

In [ ]:
import matplotlib.pyplot as plt
H = pd.DataFrame(history)
def plot(cols, labels, ylabel, fname, title, base=None):
    plt.figure(figsize=(7, 4.2))
    for c, l in zip(cols, labels):
        plt.plot(H["epoch"], H[c], marker="o", label=l)
    if base is not None:
        plt.axhline(base, ls="--", c="gray", label="before training")
    if best_epoch:
        plt.axvline(best_epoch, ls=":", c="green", label=f"best epoch ({best_epoch})")
    plt.xlabel("epoch"); plt.ylabel(ylabel); plt.title(title); plt.grid(alpha=.3); plt.legend()
    plt.tight_layout(); plt.savefig(f"{OUT}/{fname}", dpi=150); plt.show()

plot(["train_loss", "val_loss"], ["training loss", "validation loss"], "CTC loss", "loss_curve.png",
     "Training and validation loss")
plot(["val_cer"], ["validation CER"], "character error rate (%)", "cer_curve.png",
     "Validation character error rate", base_val[1])
plot(["val_word_accuracy"], ["validation word accuracy"], "words fully correct (%)", "word_accuracy_curve.png",
     "Validation word accuracy", base_val[2])
plot(["grad_norm_mean", "grad_norm_max"], ["mean gradient norm", "max gradient norm"], "gradient norm",
     "grad_norm_curve.png", "Gradient norm per epoch (before clipping at 5)")
plot([c for c in H.columns if c.startswith("grad_") and c not in ("grad_norm_mean", "grad_norm_max")],
     [c[5:] for c in H.columns if c.startswith("grad_") and c not in ("grad_norm_mean", "grad_norm_max")],
     "mean gradient norm", "grad_norm_per_layer.png", "Gradient norm per layer")
plot(["lr"], ["learning rate"], "learning rate", "learning_rate.png", "Learning rate")


## 13. Final test (writers never seen in training)
Optional: to also test on your classmates, add your `training_data.zip` from the `pages` folder as a Kaggle
dataset (**+ Add Input → Upload**) before running this cell. It is found automatically.

In [ ]:
model.load_state_dict(torch.load(f"{OUT}/best_model.pt", map_location=DEVICE))
after_test = evaluate(model, test_loader, keep_examples=40)

lines = ["Test set: SSL-HWD-Words writers never used for training or validation",
         f"{'':10}{'CER %':>10}{'words correct %':>18}{'CTC loss':>12}",
         f"{'before':10}{base_test[1]:10.2f}{base_test[2]:18.2f}{base_test[0]:12.3f}",
         f"{'after':10}{after_test[1]:10.2f}{after_test[2]:18.2f}{after_test[0]:12.3f}", ""]

# your classmates' handwriting, if training_data.zip (labels.tsv + images) was added as input
class_dirs = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/labels.tsv", recursive=True)]
class_zips = [p for p in glob.glob("/kaggle/input/**/training_data*.zip", recursive=True)]
class_items = []
for d in class_dirs:
    for line in open(f"{d}/labels.tsv", encoding="utf-8").read().splitlines()[1:]:
        fname, text, _ = line.split("\t")
        img = cv2.imread(f"{d}/{fname}", cv2.IMREAD_GRAYSCALE)
        if img is not None and text and all(c in CHARSET for c in text) and len(text) <= MAX_LABEL_LEN:
            class_items.append((img, text))
for z in class_zips if not class_items else []:
    with zipfile.ZipFile(z) as cz:
        for line in cz.read("labels.tsv").decode("utf-8").splitlines()[1:]:
            fname, text, _ = line.split("\t")
            img = cv2.imdecode(np.frombuffer(cz.read(fname), np.uint8), cv2.IMREAD_GRAYSCALE)
            if img is not None and text and all(c in CHARSET for c in text) and len(text) <= MAX_LABEL_LEN:
                class_items.append((img, text))
if class_items:
    cl = DataLoader(Words(class_items, False), batch_size=BATCH_SIZE, collate_fn=collate)
    start = HTR(len(CHARS) + 1); load_npz(start, WEIGHTS); start = start.to(DEVICE)
    b, a = evaluate(start, cl), evaluate(model, cl)
    lines += [f"Your classmates' handwriting ({len(class_items)} words, never used for training)",
              f"{'before':10}{b[1]:10.2f}{b[2]:18.2f}{b[0]:12.3f}",
              f"{'after':10}{a[1]:10.2f}{a[2]:18.2f}{a[0]:12.3f}", ""]
else:
    lines.append("(Classmates' data not added - skipped.)")
lines.append(f"Best epoch: {best_epoch}; best validation CER {best_cer:.2f}% (before: {base_val[1]:.2f}%)")
text = "\n".join(lines); print(text)
open(f"{OUT}/test_results.txt", "w").write(text + "\n")

with open(f"{OUT}/sample_predictions.txt", "w") as f:
    f.write("correct text | before training | after training\n")
    for (t, p0), (_, p1) in zip(base_test[3], after_test[3]):
        f.write(f"{t} | {p0} | {p1}\n")
print(open(f"{OUT}/sample_predictions.txt").read()[:1500])


## 14. Pack everything into one file
Then right panel → **Output** → `results` → `results.zip` → **Download** (or **Save Version** to keep it on Kaggle).

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/results", "zip", OUT)
for f in sorted(os.listdir(OUT)):
    print(f"{f:32} {os.path.getsize(f'{OUT}/{f}') / 1e6:8.2f} MB")
print("\nDownload /kaggle/working/results.zip  -> put htr_weights_best.npz into automark/model/ as htr_weights.npz")
